# Notebook 08: CCA Meta-Teaching — How This Project Practices What It Preaches

You've seen all 6 CCA patterns in the code. Now let's look at the project's own infrastructure.

The same patterns that govern `callbacks.py`, `agent_loop.py`, and `coordinator.py` also govern:

| Infrastructure | CCA Pattern Equivalent |
|---------------|------------------------|
| `.github/workflows/ci.yml` | Structured output + CI/CD flags |
| `.claude/CLAUDE.md` | Context management (project-level) |
| `.claude/skills/review-cca-compliance/SKILL.md` | Tool design (reusable, focused) |
| `.pre-commit-config.yaml` | Programmatic enforcement (Rule 1) |

**The meta-lesson:** CCA principles apply at every layer — from individual tool handlers to the CI pipeline itself.

## Setup

In [ ]:
import sys
from pathlib import Path

# Resolve the project root. Jupyter is normally launched from notebooks/, so the
# root is the parent directory; fall back to the current directory when the
# notebook is executed from the repository root.
_cwd = Path(".").resolve()
PROJECT_ROOT = _cwd if (_cwd / ".claude" / "CLAUDE.md").exists() else _cwd.parent
sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

# Verify key files exist before reading them
artifacts = [
    PROJECT_ROOT / ".claude" / "CLAUDE.md",
    PROJECT_ROOT / ".github" / "workflows" / "ci.yml",
    PROJECT_ROOT / ".claude" / "skills" / "review-cca-compliance" / "SKILL.md",
    PROJECT_ROOT / ".pre-commit-config.yaml",
]

for path in artifacts:
    status = "FOUND" if path.exists() else "MISSING"
    print(f"  [{status}] {path.relative_to(PROJECT_ROOT)}")


## Section 1: CLAUDE.md Hierarchy

The CCA exam tests whether you know the 4-level CLAUDE.md hierarchy:

| Level | Path | VCS-tracked? | Authority |
|-------|------|-------------|----------|
| Managed/Org | System directory | No | Non-overridable (cannot be excluded) |
| **Project** | `./CLAUDE.md` or `./.claude/CLAUDE.md` | **Yes** | **Team standards** |
| User | `~/.claude/CLAUDE.md` | No | Personal prefs |
| Local | `./CLAUDE.local.md` | No (you add it to `.gitignore`) | Personal override |

This project uses `.claude/CLAUDE.md` for the project level and lists `CLAUDE.local.md` in `.gitignore`. Claude Code does not ignore the local file for you.

**Why project-level matters:** When team standards live in the project file, every developer and every CI agent sees the same rules automatically. No manual coordination needed.

<div style="border-left: 4px solid #007bff; padding: 12px 16px; background: #f0f7ff; margin: 8px 0;">
<strong>CCA Exam Tip:</strong> "Team standards inconsistent" → move rules to the project-level <code>CLAUDE.md</code> (VCS-tracked). Personal preferences belong in <code>~/.claude/CLAUDE.md</code> or <code>CLAUDE.local.md</code> (never committed). Secrets belong in neither: use environment variables or a secrets store.
</div>

In [ ]:
# Read and display the project-level CLAUDE.md
claude_md_path = PROJECT_ROOT / ".claude" / "CLAUDE.md"
content = claude_md_path.read_text()
print(f"=== {claude_md_path.relative_to(PROJECT_ROOT)} ===")
print("(VCS-tracked: yes — committed to main branch)")
print(f"Character count: {len(content)}")
print()
print(content)

<div style="border-left: 4px solid #007bff; padding: 12px 16px; background: #f0f7ff; margin: 8px 0;">
<strong>CCA Exam Tip:</strong> Notice the "Architecture Rules" section above enforces rules that <em>code</em> must follow — not just guidelines. The rule "Business rules enforced in <code>callbacks.py</code>, NEVER in system prompts alone" is the same principle demonstrated in Notebooks 01 (Escalation) and 02 (Compliance).
</div>

## Section 2: CI/CD Pipeline

The CI workflow at `.github/workflows/ci.yml` is itself a CCA teaching artifact. Every `claude` CLI flag has an inline comment explaining its exam significance.

**The critical CI flags:**

| Flag | What it does | CCA Exam Signal |
|------|--------------|----------------|
| `-p` | Non-interactive mode. Required for any unattended run | Pipeline hanging → missing `-p` |
| `--bare` | Skips hooks, skills, commands, subagents, plugins, MCP servers, auto memory, and CLAUDE.md, so the run is the same on every machine. Also skips OAuth login, so `ANTHROPIC_API_KEY` must be set | Inconsistent CI → add `--bare` |
| `--output-format json` | Structured output with the text in the `result` field | Raw text is non-deterministic |
| `--allowedTools` | Pre-approves the listed tools so they run without a prompt. It does not remove the others | Least privilege: list only Read/Grep/Glob for a review |
| `--permission-mode dontAsk` | Denies anything that would otherwise prompt | Pair with `--allowedTools` for a locked-down CI run |

Two consequences worth noticing in the workflow below. `--bare` is why the API key comes from a GitHub secret rather than a login. And because `--bare` skips skill discovery, the CI job cannot invoke the project's review skill (Section 3); it inlines the same checklist in its prompt instead.

In [ ]:
# Read and display the CI workflow
ci_yml_path = PROJECT_ROOT / ".github" / "workflows" / "ci.yml"
content = ci_yml_path.read_text()
print(f"=== {ci_yml_path.relative_to(PROJECT_ROOT)} ===")
print(f"Line count: {len(content.splitlines())}")
print()
print(content)

<div style="border-left: 4px solid #dc3545; padding: 12px 16px; background: #fff5f5; margin: 8px 0;">
<strong>Anti-Pattern Warning:</strong> Using <code>claude</code> without <code>-p</code> in CI is one of the most common exam traps. Nobody is there to type, so the run waits for interactive input that never comes. Use <code>-p</code> in any automated context.
</div>

<div style="border-left: 4px solid #007bff; padding: 12px 16px; background: #f0f7ff; margin: 8px 0;">
<strong>CCA Exam Tip:</strong> On the exam, if a pipeline is 'hanging', the answer is almost always: missing <code>-p</code> flag.
</div>

In [ ]:
# Verify the CI workflow's claude invocation uses the required flags.
#
# A substring check ("--bare" in ci_content) is not proof: every flag name also
# appears in the workflow's comments, so the check would still pass if the
# command itself were deleted. Instead, strip comment lines, pull out the
# `claude ...` command, and tokenize its arguments.
import re

ci_content = (PROJECT_ROOT / ".github" / "workflows" / "ci.yml").read_text()

# 1. Drop comment lines so a flag mentioned only in a comment cannot pass.
code_lines = [line for line in ci_content.splitlines() if not line.lstrip().startswith("#")]
code_only = "\n".join(code_lines)

# 2. Extract the claude command: from `claude \` up to the opening quote of the prompt.
match = re.search(r"claude \\\n(.*?)\n\s*\"", code_only, re.DOTALL)
assert match, "no `claude` command found in ci.yml"
# Line-continuation backslashes are dropped; no argument here contains whitespace.
argv = match.group(1).replace("\\", " ").split()
print(f"claude argv: {argv}")

def has_flag(flag: str, value: str | None = None) -> bool:
    if flag not in argv:
        return False
    return value is None or argv[argv.index(flag) + 1] == value

checks = [
    (has_flag("-p"), "-p", "Non-interactive mode (required for unattended runs)"),
    (has_flag("--bare"), "--bare", "Reproducibility (no hooks, skills, MCP, or CLAUDE.md loaded)"),
    (has_flag("--output-format", "json"), "--output-format json", "Structured output for programmatic parsing"),
    (has_flag("--allowedTools", "Read,Grep,Glob"), "--allowedTools Read,Grep,Glob", "Pre-approve only read-only tools"),
    (has_flag("--permission-mode", "dontAsk"), "--permission-mode dontAsk", "Deny anything not pre-approved"),
    ("jq -r '.result'" in code_only, "jq -r '.result'", "Extract text from JSON envelope"),
    ("${{ secrets.ANTHROPIC_API_KEY }}" in code_only, "secrets.ANTHROPIC_API_KEY", "API key from secrets (never hardcoded)"),
]

print()
print("CI Workflow CCA Flag Audit:")
all_pass = True
for present, label, description in checks:
    status = "PASS" if present else "FAIL"
    all_pass = all_pass and present
    print(f"  [{status}] {label} — {description}")

print()
if all_pass:
    print("All CCA CI flags present in the claude command — workflow follows exam best practices")
else:
    print("WARNING: Some CCA flags missing from the claude command")


## Section 3: Custom Skill

`.claude/skills/review-cca-compliance/SKILL.md` is a **custom skill**: a reusable, project-specific instruction file that Claude Code discovers and loads on demand.

**Layout that Claude Code discovers:** `.claude/skills/<name>/SKILL.md`. A flat `.claude/skills/<name>.md` file is not discovered. The optional YAML frontmatter carries a `description` that tells Claude when to load the skill on its own.

**Usage:** invoke it as a slash command with the path to review as the argument:
```
/review-cca-compliance src/customer_service/agent/agent_loop.py
```
Claude can also pick it up from natural language ("audit this file for CCA anti-patterns") when the request matches the description. Slash commands work in `-p` mode too: put `/review-cca-compliance <path>` in the prompt string. They do not work with `--bare`, which skips skill discovery.

**CCA connection:** Custom skills follow the same principle as focused tools (Pattern 3). Instead of one massive general-purpose prompt, you have a targeted skill that does exactly one thing well.

In [ ]:
# Read and display the CCA compliance review skill
import re

skill_path = PROJECT_ROOT / ".claude" / "skills" / "review-cca-compliance" / "SKILL.md"
content = skill_path.read_text()
print(f"=== {skill_path.relative_to(PROJECT_ROOT)} ===")
print(f"Character count: {len(content)}")
print(f"Frontmatter present: {content.startswith('---')}")
# Numbered '### N.' headings are the checks; the output-format template also
# uses '###', so a plain count of '###' would over-count.
checks = re.findall(r"^### \d+\. (.+)$", content, re.MULTILINE)
print(f"Checks: {len(checks)}")
for check in checks:
    print(f"  - {check}")
print()
# Show the frontmatter and usage section as a sample
lines = content.splitlines()
print("\n".join(lines[:30]))
print("\n... (truncated — full skill has", len(lines), "lines)")


<div style="border-left: 4px solid #28a745; padding: 12px 16px; background: #f0fff4; margin: 8px 0;">
<strong>Why this works:</strong> The custom skill has 11 focused checks: the 6 CCA patterns from the code, plus the agentic loop, the coordinator, the CLAUDE.md hierarchy, and the CI flags. Each check has clear PASS/WARN/FAIL criteria. Compare to the Swiss Army anti-pattern (Notebook 03): 15+ tools with vague overlap vs. 5 focused tools with clear boundaries. Same principle, different layer.
</div>

<div style="border-left: 4px solid #007bff; padding: 12px 16px; background: #f0f7ff; margin: 8px 0;">
<strong>CCA Exam Tip:</strong> Custom skills in <code>.claude/skills/&lt;name&gt;/SKILL.md</code> are project-level, VCS-tracked, and shared across the team — just like <code>.claude/CLAUDE.md</code>. They're the team's shared library of reusable instructions.
</div>

## Section 4: Pre-commit Hooks (Programmatic Enforcement)

`.pre-commit-config.yaml` enforces code style on every commit — automatically, without relying on developers to remember.

**This IS CCA Principle 1:** *Programmatic enforcement beats prompt-based guidance.*

| Enforcement | Code | Prompt equivalent |
|-------------|------|------------------|
| PII redaction | `compliance_callback` regex | "Never log credit card numbers" |
| Notebook output stripped | `nbstripout` pre-commit hook | "Remember to clear outputs before committing" |
| Code style | `ruff` pre-commit hook | "Follow PEP 8 in your commits" |
| Escalation rules | `escalation_callback` | "Escalate when amount > $500" |

The pattern is identical. Code runs every time. Prompts get forgotten.

In [ ]:
# Read and display the pre-commit configuration
precommit_path = PROJECT_ROOT / ".pre-commit-config.yaml"
content = precommit_path.read_text()
print(f"=== {precommit_path.relative_to(PROJECT_ROOT)} ===")
print(f"Character count: {len(content)}")
print()
print(content)

<div style="border-left: 4px solid #28a745; padding: 12px 16px; background: #f0fff4; margin: 8px 0;">
<strong>Why this works:</strong> <code>nbstripout</code> automatically strips Jupyter notebook outputs before every commit — no developer needs to remember. <code>ruff</code> enforces code style before the commit lands. These hooks are the same architectural principle as <code>PostToolUse</code> callbacks: deterministic code running at a defined hook point, not reminders in a prompt.
</div>

<div style="border-left: 4px solid #007bff; padding: 12px 16px; background: #f0f7ff; margin: 8px 0;">
<strong>CCA Exam Tip:</strong> When you see "developers sometimes forget to X" on the exam, the answer is always a programmatic enforcement mechanism — a hook, callback, or validation gate — not an updated system prompt or team reminder.
</div>

## Section 5: Pattern Connections

Every piece of this project's infrastructure maps to a CCA pattern you've seen in the code:

In [ ]:
# Pattern connection summary
connections = [
    {
        "Infrastructure": ".pre-commit-config.yaml (nbstripout + ruff)",
        "CCA Pattern": "Programmatic Enforcement (Principle 1)",
        "Code Equivalent": "callbacks.py PostToolUse hooks",
        "Anti-Pattern": "Prompt: 'Remember to clear outputs'",
    },
    {
        "Infrastructure": "ci.yml --allowedTools Read,Grep,Glob + --permission-mode dontAsk",
        "CCA Pattern": "Tool Design (Pattern 3: focused tools)",
        "Code Equivalent": "5 focused tools vs 15+ Swiss Army",
        "Anti-Pattern": "CI agent with every tool pre-approved",
    },
    {
        "Infrastructure": "ci.yml --output-format json + jq",
        "CCA Pattern": "Structured Output (three-layer reliability)",
        "Code Equivalent": "schema-enforced EscalationRecord (tool_choice backstop)",
        "Anti-Pattern": "Parsing raw text output with string matching",
    },
    {
        "Infrastructure": ".claude/CLAUDE.md (VCS-tracked team standards)",
        "CCA Pattern": "Context Management (structured, not ad-hoc)",
        "Code Equivalent": "ContextSummary vs raw transcripts",
        "Anti-Pattern": "Team standards in Slack messages or wikis",
    },
    {
        "Infrastructure": ".claude/skills/review-cca-compliance/SKILL.md",
        "CCA Pattern": "Tool Design (focused, single-purpose)",
        "Code Equivalent": "lookup_customer vs Swiss Army tool",
        "Anti-Pattern": "One general-purpose 'do everything' prompt",
    },
    {
        "Infrastructure": "ci.yml -p flag (non-interactive mode)",
        "CCA Pattern": "Agentic Loop (deterministic termination)",
        "Code Equivalent": "stop_reason check vs content-type check",
        "Anti-Pattern": "CI hanging = waiting for input that never comes",
    },
]

# Display as a formatted table
print("Infrastructure → CCA Pattern Connections")
print("=" * 80)
for i, conn in enumerate(connections, 1):
    print(f"\n{i}. {conn['Infrastructure']}")
    print(f"   Pattern:     {conn['CCA Pattern']}")
    print(f"   Code equiv:  {conn['Code Equivalent']}")
    print(f"   Anti-pattern: {conn['Anti-Pattern']}")

## Summary

The same 6 CCA patterns that appear in the agent code appear in the project infrastructure:

| Project Infrastructure | CCA Pattern | Notebook |
|----------------------|------------|----------|
| `callbacks.py` PostToolUse | Programmatic Enforcement | NB01, NB02 |
| `definitions.py` 5 focused tools | Tool Design | NB03 |
| `system_prompts.py` cache_control | Cost Optimization | NB04 |
| `context_manager.py` ContextSummary | Context Management | NB05 |
| `coordinator.py` explicit passing | Coordinator-Subagent | NB03 |
| `EscalationRecord` schema-enforced handoff | Handoffs | NB06 |
| `.pre-commit-config.yaml` hooks | Programmatic Enforcement | **This notebook** |
| `.github/workflows/ci.yml` flags | CI/CD + Structured Output | **This notebook** |
| `.claude/CLAUDE.md` project-level | Context Management | **This notebook** |
| `.claude/skills/review-cca-compliance/SKILL.md` | Tool Design | **This notebook** |

<div style="border-left: 4px solid #007bff; padding: 12px 16px; background: #f0f7ff; margin: 8px 0;">
<strong>Key Takeaway:</strong> CCA principles aren't just code patterns — they're architectural principles that apply at every layer of a project. A team that internalizes these patterns naturally applies them to CI pipelines, git hooks, and project configuration, not just to agent code.
</div>

---

**Continue exploring:**
- Return to `07_integration.ipynb` to see all 6 patterns in a single scenario
- Invoke the custom skill: `/review-cca-compliance src/customer_service/agent/callbacks.py`
- Read `.planning/CCA-RULES.md` for the complete authoritative rule reference